In [1]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import roc_curve, roc_auc_score
import scipy.io

In [2]:
datas = scipy.io.loadmat('pima.mat')
datay = datas['y']
y = np.array(datay)
datas = datas['X']
datas = np.array(datas)

In [3]:
compact_clusters = []
meanC = [None] * len(datas)
XC = [None] * len(datas)
ddC = [None] * len(datas)
LRD = [None] * len(datas)

In [4]:
distances = squareform(pdist(datas, 'euclidean'))
meandistVal = np.mean(distances, axis=1)
maxmeandistVal = np.mean(meandistVal)

In [5]:
for x, row in enumerate(distances):
    suitable_members = np.where(row < maxmeandistVal)[0]
    compact_clusters.append((x, suitable_members))

In [6]:
valve = 0
i = 0;
for center_idx, members in compact_clusters:
  for member in members:
    if valve == 0:
      valve = valve + 1
      meanC[center_idx] = datas[center_idx].reshape(-1,1)
      XC[center_idx] = np.sum(datas[center_idx].reshape(-1,1) ** 2)
      ddC[center_idx] = 1
    else:
      valve = valve + 1
      meanC[center_idx] = ((valve-1) / valve) * meanC[center_idx] + (1 / valve) * datas[member].reshape(-1,1)
      XC[center_idx] = (((valve-1) / valve) * ((XC[center_idx] + np.sum(datas[center_idx].reshape(-1,1) ** 2))/valve))
      ddC[center_idx] = (valve) / (1 + (np.sum((datas[member] - meanC[center_idx]) ** 2) / (XC[center_idx] - np.sum(meanC[center_idx] ** 2)))) + ddC[center_idx]
  LRD[center_idx] =  ddC[center_idx]
  valve = 0
  i = i + 1

In [7]:
totLRD = 0
LOFscore = [None] * len(datas)
for center_idx, members in compact_clusters:
  for member in members:
    if member != center_idx:
      totLRD = totLRD + LRD[member]
  if (len(members) > 1):
    LOFscore[center_idx] = (totLRD/(len(members) - 1)) * (1 / LRD[center_idx])
  else:
    LOFscore[center_idx] = 10000 #outlier
  totLRD = 0

In [ ]:
aucLOF = roc_auc_score(y, LOFscore)
print(aucLOF)
print(len(LOFscore))
fprARLOFF, tprARLOFF, thresholdsARLOFF = roc_curve(y, LOFscore)
roc_aucARLOFF = aucLOF

In [ ]:
fpr, tpr, thresholds = roc_curve(y, LOFscore)
gmeans = np.sqrt(tpr * (1 - fpr))  # G-mean at each threshold
ix = np.argmax(gmeans)
best_threshold = thresholds[ix]
print("Best Threshold:", best_threshold)
print("G-mean at Best Threshold:", gmeans[ix])

# Use best threshold to make predictions
y_pred = (LOFscore >= best_threshold).astype(int)